# <font size=6><b>Lec03. [실습] 영화 유사도

# <b>데이터 로드 & 전처리

In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.download('stopwords') #불용어 목록 파일 ex. is, the, a, of..
nltk.download('punkt_tab') #문장을 단어로 쪼갤 때 쓰는 규칙 파일

# ① 데이터 로드
df = pd.read_csv(r"C:\IT\workspace_ptyhon\dl\LLM\movie\movies_metadata.csv", low_memory=False)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


* 영화 추천에 필요한 컬럼<br>
   title, overview(줄거리), vote_average, vote_count

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [4]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [5]:
# ② 필요한 컬럼만 추출
df = df[['title', 'overview', 'genres', 'vote_average', 'vote_count']].copy()

# ③ 결측값 제거 — overview 없으면 TF-IDF 불가
df = df.dropna(subset=['overview'])
df = df.reset_index(drop=True)

print(f"사용 가능한 영화 수: {len(df)}")
df.head(3)

사용 가능한 영화 수: 44512


,title,overview,genres,vote_average,vote_count
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",7.7,5415.0
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",6.9,2413.0
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",6.5,92.0


In [6]:
# ④ 텍스트 정제 함수 — lec01의 전처리를 함수 하나로 묶기
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 소문자 변환
    text = text.lower()
    # 특수기호, 숫자 제거
    text = re.sub(r'[^a-z\s]', '', text)
    # 토큰화 (단어로 쪼개기)
    tokens = word_tokenize(text)
    # 불용어 제거 + 짧은 단어(1글자) 제거
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    # 다시 문장으로 합치기 (TfidfVectorizer 입력 형식)
    return ' '.join(tokens) #리스트 원소들을 공백으로 이어붙여서 문자열로 만들기

# ⑤ 적용
df['overview_clean'] = df['overview'].apply(clean_text)

# 원본 vs 정제 비교
print("원본:", df['overview'][0][:100])
print("정제:", df['overview_clean'][0][:100])

원본: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto 
정제: led woody andys toys live happily room andys birthday brings buzz lightyear onto scene afraid losing


In [7]:
# ⑥ 확인 — 정제 후 상위 빈도 단어 확인
from collections import Counter

all_words = ' '.join(df['overview_clean']).split() #거대한 문자열 하나로 합친 후 공백 기준으로 나누기
vocab = Counter(all_words)
print("Top 20 단어:", vocab.most_common(20))

Top 20 단어: [('life', 7424), ('one', 7230), ('young', 6411), ('film', 5774), ('new', 5576), ('two', 5140), ('love', 5123), ('man', 5100), ('story', 4552), ('family', 4477), ('world', 4311), ('find', 3581), ('years', 3449), ('woman', 3368), ('time', 3268), ('get', 3030), ('father', 2861), ('finds', 2842), ('back', 2755), ('friends', 2744)]


# <b>TF-IDF 벡터화
* 전처리된 텍스트를 숫자 행렬로 바꿈

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ① TfidfVectorizer 설정
tfidf = TfidfVectorizer(
    stop_words='english',  # 불용어 한 번 더 제거 (혹시 남은 것)
    max_features=10000,    # 전체 단어 중 빈도 상위 10000개만 사용
    min_df=3,              # 3편 미만 등장 단어는 제거 (너무 희귀한 단어)
    max_df=0.8,            # 80% 이상 문서에 등장하는 단어도 제거 (너무 흔한 단어)
)

# ② 벡터화 — overview_clean 컬럼 전체를 한 번에 변환
tfidf_matrix = tfidf.fit_transform(df['overview_clean'])

print(tfidf_matrix.shape)
# (45000, 10000) → 영화 45000편 × 단어 10000개 행렬

(44512, 10000)


In [9]:
# ③ 실제로 어떤 단어들이 남았는지 확인
feature_names = tfidf.get_feature_names_out()
print(f"총 단어 수: {len(feature_names)}")
print("샘플 단어:", feature_names[100:110])
# ['abandon', 'ability', 'aboard', 'absence', 'abuse', ...]

총 단어 수: 10000
샘플 단어: ['adapted' 'add' 'added' 'addict' 'addicted' 'addiction' 'addicts'
 'adding' 'addition' 'address']


In [10]:
# 첫 5편, 첫 10개 단어만 잘라서 보기
sample_df = pd.DataFrame(
    tfidf_matrix[:5, :10].toarray(),
    columns=feature_names[:10],
    index=df['title'][:5]
)
print(sample_df) #한 영화 줄거리에 쓰이는 단어는 많아야 50~100개라서 나머지 전부 0처리 == 희소행렬

                             aaron  abandon  abandoned  abandonment  abandons  \
title                                                                           
Toy Story                      0.0      0.0        0.0          0.0       0.0   
Jumanji                        0.0      0.0        0.0          0.0       0.0   
Grumpier Old Men               0.0      0.0        0.0          0.0       0.0   
Waiting to Exhale              0.0      0.0        0.0          0.0       0.0   
Father of the Bride Part II    0.0      0.0        0.0          0.0       0.0   

                             abbott  abby  abduct  abducted  abduction  
title                                                                   
Toy Story                       0.0   0.0     0.0       0.0        0.0  
Jumanji                         0.0   0.0     0.0       0.0        0.0  
Grumpier Old Men                0.0   0.0     0.0       0.0        0.0  
Waiting to Exhale               0.0   0.0     0.0       0.0        

# <b>코사인 유사도

```python
from sklearn.metrics.pairwise import cosine_similarity

① 코사인 유사도 행렬 계산
tfidf_matrix가 (45000 x 10000) 이었으니까
cosine_sim은 (45000 x 45000)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
#양이 너무 많아서 코드 안돌어감

print(cosine_sim.shape)
# (45000, 4500 0)

print(cosine_sim[0])
# [1.0, 0.03, 0.0, 0.12, ...]
# 0번 영화와 모든 영화의 유사도 점수

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"유사도 행렬 크기: {cosine_sim.shape}")

title_to_idx = pd.Series(df.index, index=df['title']) # 제목으로 행번호 찾기

def recommend(title, top_n=10): #제목 ->행 번호
    idx = title_to_idx[title]
    sim_scores = list(enumerate(cosine_sim[idx])) #행의 유사도 점수 전부 꺼내기
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) #유사도 높은 순서로 정렬
    sim_scores = sim_scores[1:top_n+1] #맨 앞(자기자신)빼고 top-n개
    movie_indices = [i[0] for i in sim_scores] #행 번호 뽑기
    result = df[['title', 'vote_average']].iloc[movie_indices].copy() #행 번호로 제목, 평점 꺼내서 반환
    result['similarity'] = [round(i[1], 3) for i in sim_scores]
    return result

유사도 행렬 크기: (44512, 44512)


# <b>실행

In [13]:
recommend('Toy Story')

,title,vote_average,similarity
15282,Toy Story 3,7.6,0.437
2979,Toy Story 2,7.3,0.428
24316,Small Fry,6.8,0.325
17102,Group Sex,5.5,0.242
483,Malice,5.9,0.219
11367,For Your Consideration,5.9,0.202
1916,Condorman,5.6,0.195
1058,Rebel Without a Cause,7.6,0.189
38889,Ozzie,5.0,0.188
25340,"Listen, Darling",0.0,0.178


# <b>실행부분 보완
* 위에 실행은 장르가 달라도 비슷한 단어면 추천됨
* 평점이 1점이어도 유사도 높으면 추천됨

In [14]:
import ast

# ① genres 컬럼 리스트처럼 보이지만 문자열
print(df['genres'][0])
# "[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}]"

# ② 장르 이름만 뽑는 함수
def extract_genres(genres_str):
    try:
        genres = ast.literal_eval(genres_str)  # 문자열 → 딕셔너리 리스트
        return ' '.join([g['name'].lower() for g in genres])
    except:
        return ''

df['genres_clean'] = df['genres'].apply(extract_genres)

print(df['genres_clean'][0])
# 'animation comedy family'

[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]
animation comedy family


In [15]:
# ③ 줄거리 + 장르 합치기
# 장르를 여러 번 반복해서 가중치 높이기
df['combined'] = df['overview_clean'] + ' ' + df['genres_clean'] * 3
# genres_clean * 3 하면 'action crime drama action crime drama action crime drama'
# 단어가 3번 반복되니까 TF-IDF에서 더 높은 점수 받음

print(df['combined'][0])
# 'young boy lives secret life ... animation comedy animation comedy animation comedy'

led woody andys toys live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plots buzz circumstances separate buzz woody owner duo eventually learns put aside differences animation comedy familyanimation comedy familyanimation comedy family


In [ ]:
# ④ combined로 다시 벡터화
tfidf2 = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    min_df=3,
    max_df=0.8,
)

tfidf_matrix2 = tfidf2.fit_transform(df['combined'])
cosine_sim2 = cosine_similarity(tfidf_matrix2, tfidf_matrix2)

In [ ]:
# ⑤ 추천 함수에 평점 필터 추가
def recommend_v2(title, top_n=10, min_rating=6.0):
    idx = title_to_idx[title]
    sim_scores = list(enumerate(cosine_sim2[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n*3+1]  # 여유있게 30개 뽑고
    movie_indices = [i[0] for i in sim_scores]

    result = df[['title', 'vote_average']].iloc[movie_indices].copy()
    result['similarity'] = [round(i[1], 3) for i in sim_scores]

    # 평점 min_rating 이상인 것만 필터링 후 top_n개
    result = result[result['vote_average'] >= min_rating]
    result = result.head(top_n)
    return result

# ⑥ 실행
print(recommend_v2('The Dark Knight'))
print(recommend_v2('Toy Story'))

In [ ]:
print("── Step 3 (줄거리만) ──")
print(recommend('The Dark Knight'))

print("── Step 4 (장르+줄거리+평점) ──")
print(recommend_v2('The Dark Knight'))